# Route C (rat): redox enzyme genes in hepatocytes sorted by ploidy (GSE132409, Katsuda et al. 2020)

Choose **Runtime → Run all**. The data download from GEO automatically into *Colab Notebooks/route_c_rat*.

In [ ]:
#@title Cell 1 - Settings and connect Google Drive
import gzip, io, os, re, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except ImportError:
    display = print

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/route_c_rat"  #@param {type:"string"}
GSE = "GSE132409"  #@param {type:"string"}
N_RANDOM_SETS = 2000  #@param {type:"integer"}
SEED = 1  #@param {type:"integer"}
DATA_DIR = os.environ.get("ROUTE_CR_DATA", DATA_DIR)
if DATA_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(DATA_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)


class StopHere(Exception):
    """Stops the notebook with a plain-language message."""


print("Dataset:", GSE, "| folder:", DATA_DIR)

In [ ]:
#@title Cell 2 - Download the GEO series matrix and the platform annotation
BASE = os.environ.get("ROUTE_CR_GEO", "https://ftp.ncbi.nlm.nih.gov/geo")


def fetch(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"   already here: {os.path.basename(dest)}")
        return True
    r = subprocess.run(["curl", "-fsSL", "-m", "1800", "-o", dest + ".part", url], capture_output=True, text=True)
    if r.returncode != 0:
        if os.path.exists(dest + ".part"):
            os.remove(dest + ".part")
        return False
    os.replace(dest + ".part", dest)
    print(f"   downloaded {os.path.basename(dest)} ({os.path.getsize(dest) / 1e6:.1f} MB)")
    return True


def listing(url):
    r = subprocess.run(["curl", "-fsSL", "-m", "120", url], capture_output=True, text=True)
    return re.findall(r'href="([^"]+)"', r.stdout) if r.returncode == 0 else []


stub = GSE[:-3] + "nnn"
mdir = f"{BASE}/series/{stub}/{GSE}/matrix/"
names = [h for h in listing(mdir) if h.endswith("_series_matrix.txt.gz")] or [f"{GSE}_series_matrix.txt.gz"]
matrices = []
for n in names:
    dest = os.path.join(DATA_DIR, n)
    if fetch(mdir + n, dest):
        matrices.append(dest)
if not matrices:
    raise StopHere(f"Could not download the series matrix from {mdir}. Check that GEO opens in your browser and run again.")

with gzip.open(matrices[0], "rt", errors="replace") as h:
    head = h.read(200000)
gpl = re.search(r'!Series_platform_id\s+"?(GPL\d+)', head).group(1)
gstub = gpl[:-3] + "nnn" if len(gpl) > 6 else "GPLnnn"
annot = os.path.join(DATA_DIR, f"{gpl}.annot.gz")
full = os.path.join(DATA_DIR, f"{gpl}_full.txt")
got = fetch(f"{BASE}/platforms/{gstub}/{gpl}/annot/{gpl}.annot.gz", annot)
if not got:
    got = fetch(f"https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={gpl}&targ=self&form=text&view=data", full)
if not got:
    raise StopHere(f"Could not download the annotation for platform {gpl}. Send me this message.")
print("Platform:", gpl)

In [ ]:
#@title Cell 3 - Samples, ploidy labels and gene-level expression
def read_matrix(path):
    meta, rows, inside = {}, [], False
    with gzip.open(path, "rt", errors="replace") as h:
        for line in h:
            line = line.rstrip("\n")
            if line.startswith("!series_matrix_table_begin"):
                inside = True; continue
            if line.startswith("!series_matrix_table_end"):
                break
            if inside:
                rows.append(line.split("\t"))
            elif line.startswith("!Sample_"):
                k, *v = line.split("\t")
                meta.setdefault(k, []).append([x.strip('"') for x in v])
    table = pd.DataFrame(rows[1:], columns=[c.strip('"') for c in rows[0]]) if len(rows) > 1 else None
    return meta, table


samples, tables = [], []
for m in matrices:
    meta, tab = read_matrix(m)
    gsm = meta["!Sample_geo_accession"][0]
    text = [" | ".join(v[i] for vs in meta.values() for v in vs if i < len(v)) for i in range(len(gsm))]
    samples.append(pd.DataFrame({"gsm": gsm, "title": meta["!Sample_title"][0], "text": text}))
    if tab is not None:
        tables.append(tab.set_index(tab.columns[0]))
samples = pd.concat(samples, ignore_index=True)
if not tables or tables[0].shape[0] == 0:
    raise StopHere("The series matrix holds no expression values (the data are only in supplementary files). Send me this message.")
expr = pd.concat(tables, axis=1)
expr = expr.apply(pd.to_numeric, errors="coerce")
expr.index = expr.index.astype(str).str.strip('"')


def ploidy_of(s):
    m = re.search(r"(?<![\d.])(2|4|8|16)\s*-?\s*[cCnN](?![a-zA-Z])", s) or re.search(r"(?i)(di|tetra|octa)ploid", s)
    if not m:
        return np.nan
    g = m.group(1).lower()
    return {"di": 2, "tetra": 4, "octa": 8}.get(g, float(g) if g.isdigit() else np.nan)


samples["ploidy"] = [ploidy_of(t) if not np.isnan(ploidy_of(t)) else ploidy_of(x) for t, x in zip(samples.title, samples.text)]
sheet = os.path.join(DATA_DIR, "sample_labels.csv")
if samples.ploidy.isna().any() and os.path.exists(sheet):
    samples = samples.drop(columns="ploidy").merge(pd.read_csv(sheet)[["gsm", "ploidy"]], on="gsm")
    print("Using your edited sample_labels.csv")
samples[["gsm", "title", "ploidy"]].to_csv(sheet, index=False)
display(samples[["gsm", "title", "ploidy"]])
if samples.ploidy.isna().any():
    raise StopHere("Some samples have no ploidy label. Open sample_labels.csv in your Drive folder, type 2, 4 or 8 in "
                   "the ploidy column, save, and run this cell again.")
samples = samples[samples.gsm.isin(expr.columns)].reset_index(drop=True)
expr = expr[samples.gsm]
print(samples.groupby("ploidy").size().rename("samples").to_string())

# Probe -> gene symbol
if os.path.exists(annot):
    with gzip.open(annot, "rt", errors="replace") as h:
        txt = h.read()
else:
    txt = open(full, errors="replace").read()
if "!platform_table_begin" in txt:
    txt = txt.split("!platform_table_begin", 1)[1].split("!platform_table_end", 1)[0]
lines = [l for l in txt.splitlines() if l and not l.startswith(("!", "#", "^"))]
ann = pd.read_csv(io.StringIO("\n".join(lines)), sep="\t", dtype=str, on_bad_lines="skip")
idcol = ann.columns[0]
symcol = next((c for c in ann.columns if c.lower().replace("_", " ") in ("gene symbol", "symbol", "genesymbol")), None)
if symcol is None:
    symcol = next((c for c in ann.columns if "assignment" in c.lower() or "symbol" in c.lower()), None)
if symcol is None:
    raise StopHere(f"No gene symbol column in the annotation. Columns: {list(ann.columns)[:15]}. Send me this message.")
sym = ann.set_index(ann[idcol].astype(str))[symcol].astype(str)
if "assignment" in symcol.lower():
    sym = sym.str.split("//").str[1]
sym = sym.str.split("///").str[0].str.strip()
sym = sym[(sym != "") & (sym != "nan") & (sym != "---")]
if np.nanmax(expr.to_numpy()) > 100:
    expr = np.log2(expr.clip(lower=0) + 1)
    print("Values were not log-scaled: log2 applied.")
e = expr.loc[expr.index.intersection(sym.index)].copy()
e["gene"] = sym.loc[e.index].to_numpy()
e["mean"] = e[samples.gsm].mean(axis=1)
e = e.sort_values("mean", ascending=False).drop_duplicates("gene").set_index("gene")[samples.gsm]
e = e[e.notna().all(axis=1)]
# Make samples comparable: quantile normalisation (every sample gets the same distribution of values)
ranks = e.rank(axis=0, method="first").astype(int) - 1
ref = np.sort(e.to_numpy(), axis=0).mean(axis=1)
e = pd.DataFrame(ref[ranks.to_numpy()], index=e.index, columns=e.columns)
print("Samples quantile-normalised.")
# Animal / replicate number from titles such as 2c_3 (the same number = cells sorted from the same liver)
samples["animal"] = samples.title.str.extract(r"[_\s-](\d+)\s*$", expand=False)
e = e[e.mean(axis=1) > np.quantile(e.mean(axis=1), 0.25)]        # drop the 25% least expressed genes
print(f"Genes kept: {len(e):,} (most expressed probe per gene; bottom quarter by expression removed)")

In [ ]:
#@title Cell 4 - Checks: do samples group by ploidy, the cell-cycle control, and liver zonation
lp = np.log2(samples.ploidy.to_numpy())
C = np.corrcoef((e.sub(e.mean(axis=1), axis=0)).T.to_numpy()); np.fill_diagonal(C, -1)
nn = C.argmax(axis=1)
print(f"Nearest-neighbour sample has the same ploidy for {np.mean(samples.ploidy.to_numpy()[nn] == samples.ploidy.to_numpy()):.0%} of samples")
PAIRED = samples.animal.notna().all() and samples.animal.nunique() < len(samples)
if PAIRED:
    same_animal = np.mean(samples.animal.to_numpy()[nn] == samples.animal.to_numpy())
    print(f"Nearest-neighbour sample has the same replicate number for {same_animal:.0%} of samples "
          f"(chance about {1 / samples.animal.nunique():.0%}).")
    print("Samples with the same number are treated as coming from the same animal: every comparison is made within animal.")

Z = e.sub(e.mean(axis=1), axis=0).div(e.std(axis=1).replace(0, 1), axis=0)


def sample_score(genes):
    g = [x for x in genes if x in Z.index]
    return (Z.loc[g].mean(), len(g)) if len(g) >= 3 else (pd.Series(np.nan, index=Z.columns), len(g))


CHECKS = {"cell cycle (positive control: up in 8c, reported by the authors)":
          "Mki67 Top2a Ccnb1 Cdk1 Ccna2 Bub1 Aurkb Plk1 Cdc20 Ccnb2 Kif11 Cenpf Nusap1 Prc1".split(),
          "pericentral (central vein) genes": "Glul Cyp2e1 Cyp1a2 Oat Lect2 Axin2 Rhbg Cyp7a1 Slc1a2 Cyp27a1 Lgr5".split(),
          "periportal genes": "Cps1 Ass1 Arg1 Hal Sds Pck1 Cyp2f4 Gls2 Hsd17b13 Aldob Igfbp2 Hamp Cyp8b1".split()}
chk = {}
for name, gs in CHECKS.items():
    s, n = sample_score(gs); chk[name] = s
    by = s.groupby(samples.ploidy.to_numpy()).mean()
    print(f"{name} ({n} genes): " + ", ".join(f"{int(k)}c {v:+.2f}" for k, v in by.items()))
zone = (chk["pericentral (central vein) genes"] - chk["periportal genes"]).to_numpy()
r_zone = np.corrcoef(zone, lp)[0, 1]
print(f"\nZonation score (pericentral minus periportal) vs ploidy: r = {r_zone:+.2f}")
if abs(r_zone) > 0.5:
    print("Ploidy groups differ in zonation. Cell 5 therefore also reports results adjusted for the zonation score.")

In [ ]:
#@title Cell 5 - Redox enzyme sets vs ploidy, against random genes, with and without zonation adjustment
SETS = {
 "Glutathione S-transferases": "Gsta1 Gsta2 Gsta3 Gsta4 Gsta5 Gstm1 Gstm2 Gstm3 Gstm4 Gstm5 Gstm6 Gstm7 Gstp1 Gstk1 Gsto1 Gstt1 Gstt2 Gstt3 Gstz1 Mgst1 Mgst2 Mgst3",
 "Glutathione synthesis & recycling": "Gclc Gclm Gss Gsr Gpx1 Gpx4 Glrx Glrx2 Glrx3 Glrx5",
 "Thioredoxin / peroxiredoxin": "Txn1 Txnrd1 Txn2 Txnrd2 Prdx1 Prdx2 Prdx3 Prdx4 Prdx5 Prdx6 Srxn1 Txnl1",
 "H2O2 / ROS-producing enzymes": "Cyp2e1 Xdh Maoa Maob Ero1a Ero1b Acox1 Acox2 Acox3 Hao1 Hao2 Dao Aox1 Nox4",
 "NADPH supply": "G6pd Pgd Tkt Taldo1 Me1 Me2 Idh1 Idh2 Nnt Mthfd1 Aldh1l1",
 "NRF2 targets (other)": "Nqo1 Hmox1 Ephx1 Cbr1 Cbr3 Akr7a3 Akr1b8 Ftl1 Fth1 Blvrb Ugdh",
 "Superoxide dismutase & catalase": "Sod1 Sod2 Sod3 Cat",
 "CONTROL: cell cycle": " ".join(CHECKS["cell cycle (positive control: up in 8c, reported by the authors)"]),
}
present = set(e.index)
SETS = {k: [g for g in v.split() if g in present] for k, v in SETS.items()}
for k, v in SETS.items():
    print(f"{k}: {len(v)} genes found")


def slopes(X):
    """Per-gene coefficient of log2(ploidy): change per doubling of ploidy."""
    Y = e.to_numpy().T
    beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
    resid = Y - X @ beta
    dof = max(len(Y) - X.shape[1], 1)
    se = np.sqrt((resid**2).sum(0) / dof * np.linalg.inv(X.T @ X)[0, 0])
    return pd.Series(beta[0], index=e.index), pd.Series(beta[0] / np.where(se > 0, se, np.nan), index=e.index)


block = pd.get_dummies(samples.animal).astype(float).to_numpy() if PAIRED else np.ones((len(lp), 1))
X0 = np.column_stack([lp, block])                     # ploidy first, then one baseline per animal
zc = (zone - zone.mean()) / (zone.std() or 1)
X1 = np.column_stack([X0, zc])
models = {"ploidy" + (", within animal" if PAIRED else ""): X0}
if np.linalg.matrix_rank(X1) == X1.shape[1] and len(lp) > X1.shape[1] + 2:
    models["ploidy, zonation-adjusted" + (", within animal" if PAIRED else "")] = X1
level = e.mean(axis=1)
bins = pd.qcut(level.rank(method="first"), 10, labels=False)
avoid = {g for v in SETS.values() for g in v}
pools = {b: [g for g in bins.index[bins == b] if g not in avoid] for b in range(10)}
rows = []
for mname, X in models.items():
    b, t = slopes(X)
    for name, gs in SETS.items():
        if len(gs) < 3:
            continue
        obs = b.loc[gs].median()
        null = np.array([b.loc[[rng.choice(pools[bins[g]]) for g in gs]].median() for _ in range(N_RANDOM_SETS)])
        c = np.median(null)
        p = (1 + np.sum(np.abs(null - c) >= abs(obs - c))) / (N_RANDOM_SETS + 1)
        rows.append({"model": mname, "set": name, "genes": len(gs), "change_per_doubling": obs,
                     "random_median": c, "p_vs_random": p, "share_down": float((b.loc[gs] < 0).mean())})
    if mname.startswith("ploidy") and "zonation" not in mname:
        gene_b, gene_t = b, t
res = pd.DataFrame(rows)
for mname in res.model.unique():
    m = (res.model == mname) & ~res.set.str.startswith("CONTROL")
    p = res.loc[m, "p_vs_random"].to_numpy(); o = np.argsort(p)
    q = np.minimum.accumulate((p[o] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    qq = np.empty_like(p); qq[o] = np.minimum(q, 1); res.loc[m, "q_fdr"] = qq
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 12)
display(res.round(3))
if any("zonation" in m for m in models) and abs(r_zone) > 0.9:
    print(f"NOTE: zonation and ploidy are almost perfectly linked here (r = {r_zone:+.2f}). The adjusted model then cannot tell "
          "them apart, so a real ploidy effect can vanish after adjustment. Read the adjusted rows as 'not separable', not as 'no effect'.")
print("change_per_doubling: median log2 change of the set's genes per doubling of ploidy (2c -> 4c -> 8c).")
print(f"All genes: {(gene_b < 0).mean():.0%} go down with ploidy.")

In [ ]:
#@title Cell 6 - Gene by gene, compared with the mouse hepatocyte proteins, chart and saving
genes = [g for k, v in SETS.items() if not k.startswith("CONTROL") for g in v]
tab = pd.DataFrame({"change_per_doubling": gene_b.loc[genes], "t": gene_t.loc[genes],
                    "pct_of_all_genes": gene_b.rank(pct=True).loc[genes]})
grp = samples.groupby("ploidy").gsm.apply(list)
for p_, cols in grp.items():
    tab[f"mean_{int(p_)}c"] = e.loc[genes, cols].mean(axis=1)
tab["set"] = [next(k for k, v in SETS.items() if g in v) for g in genes]
pd.set_option("display.max_rows", 200)
display(tab.sort_values("t").round(2))

mouse = os.path.join(DATA_DIR, "redox_enzymes_per_protein.csv")
if os.path.exists(mouse):
    mo = pd.read_csv(mouse).set_index("gene")
    both = tab.join(mo[["ploidy_same_dna_conc", "ploidy_same_dna_conc_large"]], how="inner")
    if len(both) >= 5:
        r = both.change_per_doubling.corr(both.ploidy_same_dna_conc)
        agree = int(((both.change_per_doubling < 0) & (both.ploidy_same_dna_conc < 0)).sum())
        print(f"\nRat mRNA vs mouse protein (Lanz, 4n vs 2n small cells), {len(both)} shared enzymes: r = {r:+.2f}; "
              f"lower with ploidy in both: {agree}")
else:
    print("\n(To compare with the mouse hepatocyte proteins, copy redox_enzymes_per_protein.csv from route_c_liver into this folder.)")

fig, ax = plt.subplots(figsize=(7, 4))
main = res[~res.model.str.contains("zonation")].set_index("set")
ax.barh(main.index, main.change_per_doubling, color=["#b2432f" if v < 0 else "#2f6db2" for v in main.change_per_doubling])
for i, (s, r_) in enumerate(main.iterrows()):
    ax.text(r_.change_per_doubling, i, f"  p={r_.p_vs_random:.3f}", va="center", fontsize=8)
ax.axvline(0, color="black", lw=0.8); ax.invert_yaxis()
ax.set_xlabel("Median log2 change per doubling of ploidy"); ax.set_title(f"{GSE}: redox enzyme genes vs hepatocyte ploidy", loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(os.path.join(DATA_DIR, "chart_rat_redox_sets.png"), dpi=150); plt.show()
res.to_csv(os.path.join(DATA_DIR, "rat_redox_sets.csv"), index=False)
tab.to_csv(os.path.join(DATA_DIR, "rat_redox_genes.csv"))
print("Saved to", DATA_DIR)